# RoverDark pipeline (RoVer-CoRe, MPC)

Each section loads the cached result of one stage, shows rover-core's figures, and saves them to `outputs/<tag>/figures/`.

Build the caches first (from the repo root):
```
uv run python -m toy_example.scripts.build_grid_input
uv run python -m toy_example.scripts.build_grid_set
```

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore", module="do_mpc")

from toy_example.config import rover_dark
from toy_example.controllers.mpc import MPC
from toy_example.grids import grid_input, grid_set
from toy_example.simulation import uncertainty
from toy_example.simulation.initial_states import ROVER_DARK
from toy_example.simulation.simulator import simulate
from toy_example.visualization.common import save_fig
from toy_example.visualization.rollout import plot_rollout, rollout_title
from toy_example.visualization.slices import plot_failure, plot_grid_input, plot_grid_set

cfg = rover_dark()
sys = cfg.system
print(cfg.tag, cfg.grid.resolution, "E =", sys.error_bound)

## 1. System

In [ ]:
fig = plot_failure(sys)
save_fig(fig, cfg, "1_failure")
plt.show()

## 2. MPC
Closed-loop rollouts with the MPC itself (no grid), perfect state estimation.

In [ ]:
import time

mpc = MPC(cfg.mpc, sys)
print(f"effective obstacle margin: {mpc.margin:.3f} m")
t0 = time.perf_counter()
ro_mpc = simulate(sys, mpc, uncertainty.zero(), ROVER_DARK, dt=cfg.mpc.dt)
per_solve = (time.perf_counter() - t0) / ro_mpc.controls.size
print(f"{per_solve * 1e3:.1f} ms per MPC solve")

fig = plot_rollout(sys, ro_mpc, rollout_title("MPC", "zero", sys, ro_mpc), show_estimate=False)
save_fig(fig, cfg, "2_mpc_rollout")
plt.show()

## 3. GridInput
MPC evaluated at every grid node; the deployed controller is a nearest-node lookup.

In [ ]:
gi = grid_input.load(cfg)
print("shape", gi.values.shape, "| ω range", gi.values.min(), gi.values.max())
fig = plot_grid_input(sys, gi)
save_fig(fig, cfg, "3_grid_input")
plt.show()

## 4. GridSet
Bounds of GridInput over the nodes inside x + E (θ wraps across ±π).

In [ ]:
gs = grid_set.load(cfg)
width = gs.upper - gs.lower
print(f"width: mean {width.mean():.3f}, max {width.max():.3f}, fraction > 0: {(width > 0).mean():.3f}")
for theta, name in [(0.0, "4_grid_set_theta0"), (np.pi, "4_grid_set_thetapi")]:
    fig = plot_grid_set(sys, gs, gi, theta)
    save_fig(fig, cfg, name)
    plt.show()

## Simulation (GridInput controller): zero and random perception error

In [ ]:
for name, player in [("zero", uncertainty.zero()), ("random", uncertainty.uniform_random(sys, seed=0))]:
    ro = simulate(sys, gi, player, ROVER_DARK)
    fig = plot_rollout(sys, ro, rollout_title("MPC grid", name, sys, ro), show_estimate=name != "zero")
    save_fig(fig, cfg, f"sim_{name}")
    plt.show()

## 5. BRT — milestone 2
## 6. Worst-case simulation — milestone 2